# Notebook 2: Data Preprocessing
## Songwriter Style Analysis - Text Cleaning and Preparation

**Objective**: Clean and preprocess lyrics for analysis

**Input**: songs_data_final.csv (raw data from Notebook 1)

**Output**: songs_preprocessed.csv (cleaned data ready for analysis)

**Processing Steps**:
1. Remove duplicates
2. Clean lyrics text
3. Tokenization and normalization
4. Remove stopwords
5. Lemmatization
6. Filter by songwriter sample size

## Step 1: Import Libraries

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

# NLTK for text processing
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

print("Libraries imported successfully")

## Step 2: Download NLTK Resources

In [ ]:
# Download required NLTK data
print("Downloading NLTK resources...")
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('omw-1.4', quiet=True)

print("[SUCCESS] NLTK resources downloaded")

## Step 3: Load Raw Data

In [ ]:
# Load the collected data
print("Loading raw data...")
print("="*60)

df = pd.read_csv('data/songs_data_final.csv')

print(f"[SUCCESS] Loaded {len(df)} songs")
print(f"\nDataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

print("\nSongs per songwriter:")
print(df['target_songwriter'].value_counts())

print("\nMissing values:")
print(df.isnull().sum())

print("\n" + "="*60)

## Step 4: Remove Duplicates

In [ ]:
# Remove duplicate songs
print("Removing duplicates...")
initial_count = len(df)

# Drop exact duplicates based on title and artist
df = df.drop_duplicates(subset=['title', 'artist'], keep='first')

duplicates_removed = initial_count - len(df)
print(f"[REMOVED] {duplicates_removed} duplicate songs")
print(f"[REMAINING] {len(df)} unique songs")

## Step 5: Define Text Cleaning Functions

In [ ]:
def clean_lyrics(text):
    """
    Clean raw lyrics text
    
    Args:
        text: Raw lyrics string
    
    Returns:
        str: Cleaned lyrics
    """
    if not isinstance(text, str):
        return ""
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove [Verse], [Chorus], [Bridge] markers
    text = re.sub(r'\[.*?\]', '', text)
    
    # Remove parentheses and content
    text = re.sub(r'\(.*?\)', '', text)
    
    # Remove "Embed" at the end (Genius artifact)
    text = re.sub(r'\d*embed$', '', text)
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # Remove extra whitespace and newlines
    text = re.sub(r'\s+', ' ', text)
    
    # Remove special characters but keep apostrophes
    text = re.sub(r"[^a-zA-Z0-9\s']", '', text)
    
    # Remove leading/trailing whitespace
    text = text.strip()
    
    return text


def preprocess_lyrics(text):
    """
    Preprocess cleaned lyrics for NLP analysis
    
    Args:
        text: Cleaned lyrics string
    
    Returns:
        str: Preprocessed lyrics (tokenized, lemmatized, stopwords removed)
    """
    if not text:
        return ""
    
    # Tokenization
    tokens = word_tokenize(text)
    
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [token for token in tokens if token not in stop_words]
    
    # Remove very short tokens (1 character)
    tokens = [token for token in tokens if len(token) > 1]
    
    # Lemmatization
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    
    # Join back to string
    return ' '.join(tokens)


print("Text processing functions defined:")
print("  - clean_lyrics(): Remove noise and formatting")
print("  - preprocess_lyrics(): Tokenize, lemmatize, remove stopwords")

## Step 6: Apply Text Cleaning

In [ ]:
# Apply cleaning function
print("Cleaning lyrics text...")
print("This may take a few minutes...")
print("="*60)

df['lyrics_clean'] = df['lyrics'].apply(clean_lyrics)

print("[SUCCESS] Lyrics cleaned")

# Show example
sample_idx = 0
print("\nExample - Original vs Cleaned:")
print("-"*60)
print("ORIGINAL:")
print(df['lyrics'].iloc[sample_idx][:300])
print("\n" + "-"*60)
print("CLEANED:")
print(df['lyrics_clean'].iloc[sample_idx][:300])
print("\n" + "="*60)

## Step 7: Apply Preprocessing (Tokenization and Lemmatization)

In [ ]:
# Apply preprocessing function
print("Preprocessing lyrics (tokenization, lemmatization)...")
print("This may take several minutes...")
print("="*60)

# Track progress
total = len(df)
processed_lyrics = []

for idx, text in enumerate(df['lyrics_clean']):
    if idx % 50 == 0:
        print(f"  Progress: {idx}/{total} ({100*idx/total:.1f}%)")
    
    processed = preprocess_lyrics(text)
    processed_lyrics.append(processed)

df['lyrics_processed'] = processed_lyrics

print(f"\n[SUCCESS] All {total} songs preprocessed")

# Show example
print("\nExample - Cleaned vs Preprocessed:")
print("-"*60)
print("CLEANED:")
print(df['lyrics_clean'].iloc[sample_idx][:300])
print("\n" + "-"*60)
print("PREPROCESSED:")
print(df['lyrics_processed'].iloc[sample_idx][:300])
print("\n" + "="*60)

## Step 8: Filter by Minimum Sample Size

In [ ]:
# Filter songwriters with insufficient data
print("Filtering songwriters by minimum sample size...")
print("="*60)

MIN_SONGS_PER_SONGWRITER = 20  # Minimum songs needed for training

print(f"Minimum required: {MIN_SONGS_PER_SONGWRITER} songs per songwriter")
print("\nBefore filtering:")
songwriter_counts_before = df['target_songwriter'].value_counts()
print(songwriter_counts_before)

# Get songwriters with sufficient data
valid_songwriters = songwriter_counts_before[songwriter_counts_before >= MIN_SONGS_PER_SONGWRITER].index

# Filter dataset
df_filtered = df[df['target_songwriter'].isin(valid_songwriters)].copy()

print("\n" + "-"*60)
print("After filtering:")
songwriter_counts_after = df_filtered['target_songwriter'].value_counts()
print(songwriter_counts_after)

print("\n" + "="*60)
print(f"Songwriters kept: {len(valid_songwriters)}")
print(f"Songs kept: {len(df_filtered)}")
print(f"Songs removed: {len(df) - len(df_filtered)}")

## Step 9: Remove Empty or Too-Short Lyrics

In [ ]:
# Remove songs with very short processed lyrics
print("Removing songs with insufficient lyrics...")
print("="*60)

MIN_WORD_COUNT = 30  # Minimum words after preprocessing

# Calculate word count
df_filtered['word_count'] = df_filtered['lyrics_processed'].apply(lambda x: len(x.split()))

print(f"\nWord count statistics:")
print(f"  Mean: {df_filtered['word_count'].mean():.1f}")
print(f"  Median: {df_filtered['word_count'].median():.1f}")
print(f"  Min: {df_filtered['word_count'].min()}")
print(f"  Max: {df_filtered['word_count'].max()}")

before_count = len(df_filtered)
df_filtered = df_filtered[df_filtered['word_count'] >= MIN_WORD_COUNT].copy()
after_count = len(df_filtered)

print(f"\n[REMOVED] {before_count - after_count} songs with < {MIN_WORD_COUNT} words")
print(f"[REMAINING] {after_count} songs")

print("\n" + "="*60)

## Step 10: Save Preprocessed Data

In [ ]:
# Save the preprocessed dataset
print("Saving preprocessed data...")
print("="*60)

output_file = 'data/songs_preprocessed.csv'
df_filtered.to_csv(output_file, index=False)

print(f"[SAVED] Preprocessed data saved to {output_file}")

# Print final summary
print("\nFinal Dataset Summary:")
print("-"*60)
print(f"Total songs: {len(df_filtered)}")
print(f"Total songwriters: {df_filtered['target_songwriter'].nunique()}")
print(f"\nSongs per songwriter:")
print(df_filtered['target_songwriter'].value_counts())

print("\n" + "="*60)
print("PREPROCESSING NOTEBOOK COMPLETE")
print("="*60)
print(f"\nNext Step: Run 03_exploratory_analysis.ipynb")

In [ ]:
# Display sample of preprocessed data
print("Sample of Preprocessed Data:")
print("="*60)
df_filtered[['title', 'artist', 'target_songwriter', 'word_count']].head(10)